# Multi-Agent RAG for Flight Passenger Queries

**Author:** Katherine Soto | **Purpose:** AXA Tech Lead interview demo

This notebook implements a multi-agent RAG system using **Google Gemini** that:
1. Detects ambiguous queries with a **Clarification Agent**
2. Retrieves relevant docs from a **FAISS vector DB**
3. Generates grounded answers with a **Generation Agent**
4. Validates output quality with an **LLM-as-Judge**

The domain is **flight passenger queries** — baggage rules, schedules, in-flight services.

## 1. Setup

In [1]:
# !pip install -q google-generativeai faiss-cpu numpy

In [2]:
# # Run this to see your available embedding models
# import google.generativeai as genai

# for model in genai.list_models():
#     if 'embed' in model.name.lower():
#         print(f'{model.name}  →  {model.supported_generation_methods}')

In [3]:
# test = genai.embed_content(
#     model='models/gemini-embedding-001',
#     content='hello',
#     task_type='RETRIEVAL_DOCUMENT'
# )
# print(f'Embedding dim: {len(test["embedding"])}')

In [ ]:
import os
import json
import numpy as np
import faiss
import google.generativeai as genai
from typing import List, Dict, Tuple
from dataclasses import dataclass

# ──── SET YOUR GOOGLE API KEY HERE ────
# Option A: paste it directly (don't push to GitHub!)
GOOGLE_API_KEY = 'here'  

# Option B (safer): export GOOGLE_API_KEY='...' in your terminal first, then:
# GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')

genai.configure(api_key=GOOGLE_API_KEY)

# Model config
# EMBEDDING_MODEL = 'models/text-embedding-004'  # Google's latest embedding model
EMBEDDING_MODEL = 'models/gemini-embedding-001'
EMBEDDING_DIM = 3072  # gemini-embedding-001 outputs 3072 dims
LLM_MODEL = 'gemini-flash-latest'              # 2.0-flash free-tier quota=0; this alias has quota
# EMBEDDING_DIM = 768                             # text-embedding-004 outputs 768 dims

print(f'Configured with embedding: {EMBEDDING_MODEL}, LLM: {LLM_MODEL}')

Configured with embedding: models/gemini-embedding-001, LLM: gemini-flash-latest


/var/folders/5l/h4rtlcqd1yq5byc4f042mtg80000gn/T/ipykernel_53677/1292634684.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


### What I'm doing here — the infrastructure layer

This first block is my **infrastructure setup**, and I keep it deliberately thin. My mental model for any RAG system is three separable concerns: an **embedding model** (turns text into vectors for retrieval), a **generation model** (turns retrieved context into an answer), and the **client/config** that wires me to those hosted endpoints. I keep them as separate constants (`EMBEDDING_MODEL`, `LLM_MODEL`, `EMBEDDING_DIM`) precisely because in a real deployment each one is a swappable, independently-versioned piece of managed infrastructure.

A note on **model serving**: I'm calling hosted inference endpoints, not running weights myself. That means I don't own the GPUs, but I *do* own the failure modes that come with a shared multi-tenant API — quota, rate limits, region availability and model deprecation. I chose `gemini-flash-latest` (a moving alias) over a pinned version on purpose after hitting a hard `quota: 0` wall on the older pinned name; treating the model id itself as configuration is part of designing for a serving layer I don't control.

> **Production mapping (AXA):** this exact separation is why the port to Azure is mechanical — `EMBEDDING_MODEL` → Azure OpenAI `text-embedding-3`, `LLM_MODEL` → Azure OpenAI GPT-4o, and the client config → an Azure endpoint + key in Key Vault instead of an inline string.

## 2. Knowledge Base — Flight Policies

In production these would come from airline PDFs. For the demo we use a synthetic KB.

In [5]:
KNOWLEDGE_BASE = [
    {
        'id': 'BAG-001',
        'category': 'baggage',
        'text': 'Carry-on baggage must not exceed 55x40x23cm and 8kg. One personal item (backpack, purse) up to 40x30x15cm is also allowed. Oversized items must be checked in.'
    },
    {
        'id': 'BAG-002',
        'category': 'baggage',
        'text': 'Liquids in carry-on must be in containers of 100ml or less, all fitting in a single transparent resealable bag of maximum 1 liter. Baby food and prescribed medications are exempt.'
    },
    {
        'id': 'BAG-003',
        'category': 'baggage',
        'text': 'Laptops, tablets, and e-readers can be carried in carry-on baggage. During security screening they must be removed and placed in a separate tray. Power banks up to 100Wh are allowed in carry-on only, not checked baggage.'
    },
    {
        'id': 'BAG-004',
        'category': 'baggage',
        'text': 'Sharp objects including scissors with blades over 6cm, knives, and razor blades are prohibited in carry-on. They must be packed in checked baggage.'
    },
    {
        'id': 'BAG-005',
        'category': 'baggage',
        'text': 'Sporting equipment such as golf clubs, skis, and bicycles requires special handling. Contact the airline at least 48 hours before departure to arrange transport. Additional fees apply.'
    },
    {
        'id': 'FL-001',
        'category': 'schedule',
        'text': 'Flight AF1234 from Barcelona to Paris departs daily at 07:15 CET from Terminal 1, gate B12. Boarding starts 45 minutes before departure. Flight duration is approximately 2 hours.'
    },
    {
        'id': 'FL-002',
        'category': 'schedule',
        'text': 'Flight IB2050 from Barcelona to Madrid departs daily at 09:30 CET from Terminal 1, gate A5. Boarding starts 40 minutes before departure. Flight duration is approximately 1 hour 15 minutes.'
    },
    {
        'id': 'FL-003',
        'category': 'schedule',
        'text': 'Flight LH1811 from Barcelona to Frankfurt departs daily at 14:20 CET from Terminal 1, gate C8. Boarding starts 45 minutes before departure. Flight duration is approximately 2 hours 30 minutes.'
    },
    {
        'id': 'SVC-001',
        'category': 'service',
        'text': 'Onboard meals are complimentary on flights over 3 hours. On shorter flights, snacks and beverages are available for purchase. Special meals (vegetarian, kosher, gluten-free) must be requested at least 24 hours before departure.'
    },
    {
        'id': 'SVC-002',
        'category': 'service',
        'text': 'Wi-Fi is available on most long-haul flights for a fee. Prices start at 5 EUR for messaging-only access and go up to 20 EUR for full internet. Payment accepted in EUR, USD, or by credit card.'
    },
    {
        'id': 'SVC-003',
        'category': 'service',
        'text': 'Passengers traveling with infants under 2 years old receive a complimentary bassinet on request for long-haul flights. Available on a first-come, first-served basis. Reserve at check-in or online 48 hours before departure.'
    },
    {
        'id': 'SVC-004',
        'category': 'service',
        'text': 'Passengers with reduced mobility can request wheelchair assistance at least 48 hours before departure. Service includes assistance from check-in through boarding and disembarking at destination.'
    }
]

print(f'Loaded {len(KNOWLEDGE_BASE)} knowledge chunks')
print(f'Categories: {set(c["category"] for c in KNOWLEDGE_BASE)}')

Loaded 12 knowledge chunks
Categories: {'schedule', 'service', 'baggage'}


### The knowledge base — my corpus and chunking strategy

Here I define the **corpus**: the ground truth the system is allowed to speak from. In production this is the hardest and most expensive part of the infrastructure — document ingestion pipelines, OCR, PDF parsing, deduplication — so for the demo I hand-author clean synthetic chunks to isolate the *retrieval and reasoning* problem from the *ingestion* problem.

Two deliberate design choices from an infra standpoint:
- **Chunk granularity.** Each entry is one self-contained policy fact, not a whole document. Chunk size is the single biggest lever on retrieval quality — too big and the embedding gets diluted, too small and I lose context. One-fact chunks give me sharp embeddings and clean citations.
- **Metadata.** Every chunk carries a stable `id` and a `category`. The `id` is what lets me do **citation-level traceability** later (the generator cites `[BAG-003]`), and `category` is the hook a production system would use for metadata-filtered retrieval. Stable ids are the backbone of an auditable RAG system.

## 3. Vector Store — FAISS + Google Embeddings

In [6]:
def get_embedding(text: str) -> np.ndarray:
    """Convert text to embedding vector using Google text-embedding-004."""
    result = genai.embed_content(
        model=EMBEDDING_MODEL,
        content=text,
        task_type='RETRIEVAL_DOCUMENT'
    )
    return np.array(result['embedding'], dtype=np.float32)


def get_query_embedding(text: str) -> np.ndarray:
    """Embed a query (uses RETRIEVAL_QUERY task type for better retrieval)."""
    result = genai.embed_content(
        model=EMBEDDING_MODEL,
        content=text,
        task_type='RETRIEVAL_QUERY'
    )
    return np.array(result['embedding'], dtype=np.float32)


class VectorStore:
    """Simple FAISS-based vector store for retrieval."""
    def __init__(self, dim: int = EMBEDDING_DIM):
        self.index = faiss.IndexFlatIP(dim)  # Inner product = cosine on normalized vectors
        self.chunks = []

    def add(self, chunks: List[Dict]):
        """Embed and add chunks to the index."""
        vectors = []
        for chunk in chunks:
            emb = get_embedding(chunk['text'])
            emb = emb / np.linalg.norm(emb)  # normalize for cosine similarity
            vectors.append(emb)
            self.chunks.append(chunk)
        vectors = np.stack(vectors)
        self.index.add(vectors)
        print(f'Indexed {len(chunks)} chunks. Total: {self.index.ntotal}')

    def search(self, query: str, top_k: int = 3) -> List[Tuple[Dict, float]]:
        """Return top-K most similar chunks with scores."""
        q_emb = get_query_embedding(query)
        q_emb = q_emb / np.linalg.norm(q_emb)
        scores, indices = self.index.search(q_emb.reshape(1, -1), top_k)
        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx == -1:
                continue
            results.append((self.chunks[idx], float(score)))
        return results


# Build the index
vector_store = VectorStore()
vector_store.add(KNOWLEDGE_BASE)

Indexed 12 chunks. Total: 12


### Vector store — the retrieval infrastructure

This is the **retrieval backbone**. Conceptually I'm building a tiny vector database: I embed every chunk into a 3072-dim space, normalise the vectors, and index them in FAISS so I can do nearest-neighbour search at query time.

The theory I care about here:
- **Why normalise + inner product.** I L2-normalise every vector and use `IndexFlatIP` (inner product). On unit vectors, inner product *is* cosine similarity — so I get semantic closeness while keeping the fast, simple index.
- **Asymmetric embeddings.** I use `task_type='RETRIEVAL_DOCUMENT'` when indexing and `'RETRIEVAL_QUERY'` when searching. Documents and questions are phrased differently ("Laptops can be carried…" vs "can I bring my laptop?"), and telling the embedding model which side it's embedding measurably tightens retrieval. This asymmetry is easy to forget and a real quality lever.
- **`IndexFlatIP` is honest about scale.** Flat = exact brute-force search, perfect for 12 chunks. It's O(n) per query and will not scale, which is exactly why production swaps this for a managed ANN service (Azure AI Search, pgvector, Pinecone) with approximate indexes. I'm keeping the retrieval *interface* the same so that swap is a one-class change.

## 4. Helper — call Gemini

A reusable wrapper for all LLM calls.

In [7]:
import time
from google.api_core.exceptions import ResourceExhausted

def call_gemini(prompt: str, temperature: float = 0.0, json_mode: bool = False, max_retries: int = 3) -> str:
    """Call Gemini. Retry only on 429, waiting the API's suggested delay."""
    model = genai.GenerativeModel(
        model_name=LLM_MODEL,
        generation_config=genai.GenerationConfig(
            temperature=temperature,
            response_mime_type='application/json' if json_mode else 'text/plain',
        )
    )
    for attempt in range(max_retries):
        try:
            return model.generate_content(prompt).text
        except ResourceExhausted as e:
            delay = getattr(e, 'retry_delay', None)
            wait = delay.seconds if delay else 2 ** attempt * 5
            if attempt == max_retries - 1:
                raise
            print(f'429 rate-limited. Waiting {wait}s (attempt {attempt+1}/{max_retries})...')
            time.sleep(wait)


### The LLM wrapper — designing for an API I don't control

Every LLM call in this notebook goes through this one function, on purpose. Centralising inference gives me a single place to own the realities of a **shared, rate-limited serving layer**.

The important design decision is the **retry-with-backoff on `429`**. My first version blindly `sleep(60)` before every call — which was both too slow on the happy path and useless when the real problem was a `quota: 0` on a dead model id. The correct pattern, and the one here, is: *try first, and only back off when the API actually tells me to* — reading the server's own `retry_delay` and falling back to exponential backoff. This is standard resilience engineering for any hosted-inference dependency: treat throttling as an expected signal, not an exception.

> This function is also my natural seam for **observability** — in production I'd wrap it with latency timing, token counting and structured logging, because you cannot operate an LLM system you can't measure.

## 5. Agent 1 — Clarification Agent

**The key novelty.** Before retrieval, check if the query is answerable.
Detects missing entities like:
- "can I bring this?" → missing item name
- "when does my flight leave?" → missing flight number

In [ ]:
# for model in genai.list_models():
#     if 'generateContent' in model.supported_generation_methods:
#         print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-

In [ ]:
# pip install --upgrade google-generativeai

I0805 22:14:10.098561 4806174 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(90, generation: 1)
I0805 22:14:10.098835 4806174 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(92, generation: 1)
Note: you may need to restart the kernel to use updated packages.


In [10]:
# # Test directo
# # model = genai.GenerativeModel('gemini-2.0-flash')
# # response = model.generate_content('Say hello in JSON format: {"greeting": "hello"}')
# # print(response.text)

# # Test con nombre completo
# model = genai.GenerativeModel('models/gemini-2.0-flash')
# response = model.generate_content('Say hello')
# print(response.text)

In [11]:
# import requests

# # Test directo con REST API (sin gRPC)
# url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?key={GOOGLE_API_KEY}"
# response = requests.post(url, json={
#     "contents": [{"parts": [{"text": "Say hello"}]}]
# })
# print(response.status_code)
# print(response.json()['candidates'][0]['content']['parts'][0]['text'])

In [12]:
CLARIFICATION_PROMPT = """You are a clarification agent for a flight assistant. Your job is to decide if a passenger query has enough information to be answered from a knowledge base of flight policies, baggage rules, schedules, and services.

Analyze the query and return a JSON object with:
- "clarity_score": float from 0.0 (completely ambiguous) to 1.0 (perfectly clear)
- "missing_entities": list of what's missing (e.g. ["item_name", "flight_number"])
- "clarifying_questions": list of specific questions to ask the user (empty if clarity_score >= 0.7)
- "reasoning": one sentence explaining the score

Examples:

Query: "can I bring this?"
Output: {"clarity_score": 0.2, "missing_entities": ["item_name"], "clarifying_questions": ["What item are you asking about?", "Do you mean carry-on or checked baggage?"], "reasoning": "The item being referenced is not specified."}

Query: "when does AF1234 depart?"
Output: {"clarity_score": 0.95, "missing_entities": [], "clarifying_questions": [], "reasoning": "Flight number and question are both specified."}

Now analyze this query:
Query: \"{query}\"

Return only the JSON object, no other text."""


@dataclass
class ClarificationResult:
    clarity_score: float
    missing_entities: List[str]
    clarifying_questions: List[str]
    reasoning: str


def clarification_agent(query: str) -> ClarificationResult:
    """Analyze query and return clarification decision."""
    prompt = CLARIFICATION_PROMPT.replace('{query}', query)
    response_text = call_gemini(prompt, temperature=0.0, json_mode=True)
    data = json.loads(response_text)
    return ClarificationResult(
        clarity_score=data['clarity_score'],
        missing_entities=data.get('missing_entities', []),
        clarifying_questions=data.get('clarifying_questions', []),
        reasoning=data.get('reasoning', '')
    )


# Test the clarification agent
test_queries = [
    'can I bring this?',
    'when does AF1234 depart?',
    'is wifi expensive?',
    'what time does my flight leave?',
]

print('CLARIFICATION AGENT TESTS')
print('=' * 50)
for q in test_queries:
    result = clarification_agent(q)
    status = 'CLEAR' if result.clarity_score >= 0.7 else 'AMBIGUOUS'
    print(f'\nQuery: "{q}"')
    print(f'  Score: {result.clarity_score:.2f} [{status}]')
    print(f'  Reasoning: {result.reasoning}')
    if result.clarifying_questions:
        print(f'  Would ask: {result.clarifying_questions}')

CLARIFICATION AGENT TESTS

Query: "can I bring this?"
  Score: 0.20 [AMBIGUOUS]
  Reasoning: The item being referenced is not specified.
  Would ask: ['What item are you asking about?', 'Are you planning to take this in your carry-on or checked baggage?']

Query: "when does AF1234 depart?"
  Score: 0.95 [CLEAR]
  Reasoning: The query explicitly includes the flight number and clearly asks for its departure time.

Query: "is wifi expensive?"
  Score: 0.50 [AMBIGUOUS]
  Reasoning: Wi-Fi availability and pricing vary depending on the specific airline and route.
  Would ask: ['Which airline or flight number are you inquiring about?']

Query: "what time does my flight leave?"
  Score: 0.30 [AMBIGUOUS]
  Reasoning: The query asks for a departure time but does not provide a flight number or booking reference.
  Would ask: ['What is your flight number or booking confirmation code?']


### Agent 1 — Clarification: a guardrail *before* retrieval

This is the piece I'm most proud of architecturally. Standard RAG silently assumes every query is well-formed and answerable — but real passengers ask *"can I bring this?"* with no antecedent. If I embed that and retrieve, I get confidently-wrong garbage.

So before spending any retrieval or generation budget, I run a **clarification agent** that scores query answerability (0–1), names the missing entities, and generates the questions to ask back. Theoretically this is an **input guardrail**: I'm moving quality control to the cheapest possible point in the pipeline. It's the same principle as validating input at the edge of a service instead of letting bad data propagate downstream.

*(The commented cells just above — listing models, the pip upgrade, the raw REST probe that returned 429 — are my actual debugging trail from diagnosing the quota problem. I'm leaving them in for transparency rather than pretending the path was clean.)*

## 6. Agent 2 — Retrieval Agent

In [13]:
def retrieval_agent(query: str, top_k: int = 3) -> List[Dict]:
    """Retrieve top-K most relevant chunks for the query."""
    results = vector_store.search(query, top_k=top_k)
    return [
        {
            'chunk_id': chunk['id'],
            'category': chunk['category'],
            'text': chunk['text'],
            'score': round(score, 4)
        }
        for chunk, score in results
    ]


# Test retrieval
print('RETRIEVAL AGENT TEST')
print('=' * 50)
results = retrieval_agent('can I bring a laptop in my carry-on?')
for r in results:
    print(f'[{r["score"]:.3f}] {r["chunk_id"]} ({r["category"]})')
    print(f'  {r["text"][:100]}...')
    print()

RETRIEVAL AGENT TEST
[0.777] BAG-003 (baggage)
  Laptops, tablets, and e-readers can be carried in carry-on baggage. During security screening they m...

[0.691] BAG-001 (baggage)
  Carry-on baggage must not exceed 55x40x23cm and 8kg. One personal item (backpack, purse) up to 40x30...

[0.674] BAG-002 (baggage)
  Liquids in carry-on must be in containers of 100ml or less, all fitting in a single transparent rese...



### Agent 2 — Retrieval: turning a question into evidence

The retrieval agent is a thin wrapper over the vector store: embed the query, pull the **top-k** most similar chunks, return them with their similarity scores. Two things I want to flag theoretically: I return the **scores**, not just the text, because those scores are a signal an out-of-domain query would expose (low max score = weak grounding); and top-k is the classic **precision/recall dial** — too small and I miss the answer, too large and I flood the generator with noise and cost. k=3 is my balance for this corpus.

## 7. Agent 3 — Generation Agent

In [14]:
GENERATION_PROMPT = """You are a flight assistant. Answer the passenger's question using ONLY the information in the provided sources.

Rules:
1. Base your answer strictly on the sources. Do not add outside knowledge.
2. Cite the source IDs (e.g. [BAG-003]) after any specific claim.
3. If the sources do not fully answer the question, say so explicitly.
4. Keep the answer concise and passenger-friendly.

Passenger question: {query}

Sources:
{sources}

Answer:"""


def generation_agent(query: str, retrieved_chunks: List[Dict]) -> str:
    """Generate answer grounded in retrieved chunks."""
    sources_text = '\n'.join(
        f'[{c["chunk_id"]}] {c["text"]}' for c in retrieved_chunks
    )
    prompt = GENERATION_PROMPT.replace('{query}', query).replace('{sources}', sources_text)
    return call_gemini(prompt, temperature=0.2)

### Agent 3 — Generation: grounded answering, not free recall

This is where retrieval becomes an answer. The whole point of RAG is here: I force the model to answer **only from the retrieved sources** and to **cite the chunk ids** behind each claim. That instruction is what separates a *grounded* system from a plausible-sounding hallucination machine — I'm deliberately trading the model's parametric world-knowledge for traceable, source-backed statements. The `[BAG-003]`-style citations aren't cosmetic; they're the audit trail that makes the answer verifiable, which is non-negotiable in a regulated domain like insurance.

## 8. Agent 4 — LLM-as-Judge

Evaluates every answer on 3 dimensions before returning it to the user:
- **Faithfulness:** does the answer stay within the sources? (no hallucination)
- **Relevance:** does the answer address the question?
- **Groundedness:** are citations correct?

In [15]:
JUDGE_PROMPT = """You are evaluating the quality of a flight assistant's answer. Rate each dimension from 0.0 to 1.0.

Query: {query}

Sources provided:
{sources}

Answer generated:
{answer}

Evaluate:
1. faithfulness (0-1): does the answer stay within what the sources say? Penalize claims not in the sources.
2. relevance (0-1): does the answer address the query directly?
3. groundedness (0-1): are citations present and correct?

Return a JSON object with keys: faithfulness, relevance, groundedness, verdict ("PASS" if all >= 0.7, else "FAIL"), and reasoning (one sentence)."""


@dataclass
class JudgeResult:
    faithfulness: float
    relevance: float
    groundedness: float
    verdict: str
    reasoning: str


def llm_judge(query: str, retrieved_chunks: List[Dict], answer: str) -> JudgeResult:
    """Evaluate the answer's quality."""
    sources_text = '\n'.join(
        f'[{c["chunk_id"]}] {c["text"]}' for c in retrieved_chunks
    )
    prompt = JUDGE_PROMPT.replace('{query}', query).replace('{sources}', sources_text).replace('{answer}', answer)
    response_text = call_gemini(prompt, temperature=0.0, json_mode=True)
    data = json.loads(response_text)
    return JudgeResult(
        faithfulness=data['faithfulness'],
        relevance=data['relevance'],
        groundedness=data['groundedness'],
        verdict=data['verdict'],
        reasoning=data.get('reasoning', '')
    )

### Agent 4 — LLM-as-Judge: the evaluation layer

Even with good retrieval, generation can drift. So every answer is scored by a second LLM call on three axes: **faithfulness** (does it stay inside the sources?), **relevance** (does it answer the question?) and **groundedness** (are the citations real and correct?). This is the **automated evaluation / online-guardrail** layer of the infrastructure — the component that lets the system flag its own low-confidence outputs for human review instead of shipping them blind.

I'm candid about the limits of this as built: the judge shares a model family with the generator (correlated blind spots), and it emits its own verdict which I should *not* fully trust — in the smoke test it once returned a numeric verdict instead of `PASS`/`FAIL`, so the robust design computes the pass/fail threshold in my own code and reserves the LLM for the scores. Groundedness in particular is better done as a deterministic check (are the cited ids ⊆ the retrieved ids?) than left to the model. These are exactly the hardening steps I'd take next.

## 9. Orchestrator — Full pipeline

Runs all 4 agents in sequence with early-exit logic:
- If clarity < threshold → return clarifying questions (skip retrieval)
- Otherwise → retrieve → generate → judge

In [16]:
CLARITY_THRESHOLD = 0.7


def answer_query(query: str, verbose: bool = True) -> Dict:
    """Full multi-agent RAG pipeline."""
    result = {'query': query, 'stages': []}

    # ── STAGE 1: Clarification ──
    if verbose:
        print(f'\n{"=" * 60}')
        print(f'Query: "{query}"')
        print(f'{"=" * 60}')
        print('\n[STAGE 1] Clarification agent...')

    clarification = clarification_agent(query)
    result['stages'].append({
        'stage': 'clarification',
        'clarity_score': clarification.clarity_score,
        'reasoning': clarification.reasoning
    })

    if verbose:
        status = 'CLEAR' if clarification.clarity_score >= CLARITY_THRESHOLD else 'AMBIGUOUS'
        print(f'  Score: {clarification.clarity_score:.2f} [{status}]')
        print(f'  Reasoning: {clarification.reasoning}')

    if clarification.clarity_score < CLARITY_THRESHOLD:
        if verbose:
            print(f'  --> Below threshold ({CLARITY_THRESHOLD}). Asking for clarification.')
        result['action'] = 'ask_clarification'
        result['clarifying_questions'] = clarification.clarifying_questions
        result['final_response'] = (
            'I need a bit more information to help you:\n' +
            '\n'.join(f'  - {q}' for q in clarification.clarifying_questions)
        )
        if verbose:
            print(f'\nFinal response:')
            print(result['final_response'])
        return result

    # ── STAGE 2: Retrieval ──
    if verbose:
        print('\n[STAGE 2] Retrieval agent...')

    retrieved = retrieval_agent(query, top_k=3)
    result['stages'].append({
        'stage': 'retrieval',
        'chunks': [{'id': c['chunk_id'], 'score': c['score']} for c in retrieved]
    })

    if verbose:
        for c in retrieved:
            print(f'  [{c["score"]:.3f}] {c["chunk_id"]}: {c["text"][:80]}...')

    # ── STAGE 3: Generation ──
    if verbose:
        print('\n[STAGE 3] Generation agent...')

    answer = generation_agent(query, retrieved)
    result['stages'].append({'stage': 'generation', 'answer': answer})

    if verbose:
        print(f'  Answer: {answer}')

    # ── STAGE 4: LLM-as-Judge ──
    if verbose:
        print('\n[STAGE 4] LLM-as-Judge...')

    judge = llm_judge(query, retrieved, answer)
    result['stages'].append({
        'stage': 'judge',
        'faithfulness': judge.faithfulness,
        'relevance': judge.relevance,
        'groundedness': judge.groundedness,
        'verdict': judge.verdict,
        'reasoning': judge.reasoning
    })

    if verbose:
        print(f'  Faithfulness:  {judge.faithfulness:.2f}')
        print(f'  Relevance:     {judge.relevance:.2f}')
        print(f'  Groundedness:  {judge.groundedness:.2f}')
        print(f'  Verdict:       {judge.verdict}')
        print(f'  Reasoning:     {judge.reasoning}')

    result['action'] = 'answer'
    result['final_response'] = answer

    if judge.verdict == 'FAIL':
        result['warning'] = 'Judge flagged this answer — human review recommended.'
        if verbose:
            print(f'\n  WARNING: {result["warning"]}')

    return result

### Orchestrator — composing the agents into one pipeline

This is the **control plane** that turns four independent agents into a system. The key architectural idea is **early-exit routing**: if the clarification agent scores the query below threshold, I stop immediately and ask for clarification — I never spend retrieval, generation or judge budget on a query I know is unanswerable. Otherwise I run the full retrieve → generate → judge chain and attach a warning if the judge fails.

I also accumulate a `stages` trace on every run. That trace is deliberate: in production it becomes my **observability record** — per-stage scores and decisions I can log, monitor for drift, and replay. An agentic pipeline you can't trace is one you can't debug.

## 10. Demo — Run the scenarios

**Scenario 1:** Ambiguous query → clarification agent kicks in  
**Scenario 2:** Clear query → full pipeline runs  
**Scenario 3:** Flight schedule → clear and specific  
**Scenario 4:** Ambiguous flight → asks for flight number  
**Scenario 5:** Out-of-domain → should hedge

In [17]:
# Scenario 1: Ambiguous — should trigger clarification
_ = answer_query('can I bring this?')


Query: "can I bring this?"

[STAGE 1] Clarification agent...
  Score: 0.20 [AMBIGUOUS]
  Reasoning: The item being asked about and the baggage type are completely unspecified.
  --> Below threshold (0.7). Asking for clarification.

Final response:
I need a bit more information to help you:
  - What specific item are you referring to?
  - Are you planning to put it in your carry-on or checked baggage?


**Scenario 1 — ambiguous query, clarification fires.** `"can I bring this?"` scores **0.20** and the pipeline exits at Stage 1 without ever touching retrieval. Exactly the intended behaviour: instead of hallucinating about an unnamed item, the system asks *what item* and *carry-on or checked*. This is my input guardrail earning its place — the cheapest possible intervention point.

In [18]:
# Scenario 2: Clear — should run full pipeline
_ = answer_query('can I bring a laptop in my carry-on?')


Query: "can I bring a laptop in my carry-on?"

[STAGE 1] Clarification agent...
  Score: 0.95 [CLEAR]
  Reasoning: The query clearly specifies both the item (laptop) and the baggage type (carry-on).

[STAGE 2] Retrieval agent...
  [0.777] BAG-003: Laptops, tablets, and e-readers can be carried in carry-on baggage. During secur...
  [0.691] BAG-001: Carry-on baggage must not exceed 55x40x23cm and 8kg. One personal item (backpack...
  [0.674] BAG-002: Liquids in carry-on must be in containers of 100ml or less, all fitting in a sin...

[STAGE 3] Generation agent...
  Answer: Yes, you can bring a laptop in your carry-on baggage [BAG-003]. 

Please keep in mind that during security screening, you must remove your laptop from your bag and place it in a separate tray [BAG-003]. Additionally, your carry-on baggage must not exceed 55x40x23 cm and 8 kg [BAG-001].

[STAGE 4] LLM-as-Judge...
  Faithfulness:  1.00
  Relevance:     1.00
  Groundedness:  1.00
  Verdict:       PASS
  Reasoning:     T

**Scenario 2 — clear query, full pipeline runs.** `"can I bring a laptop in my carry-on?"` scores **0.95**, retrieves BAG-003 (0.78) as the top hit, and the generator produces a cited, source-grounded answer. The judge returns faithfulness/relevance/groundedness = **1.00** and **PASS**. This is the happy path end-to-end, and note the citations `[BAG-003]`/`[BAG-001]` trace every claim back to the corpus.

In [20]:
# Scenario 3: Schedule query — clear and specific
_ = answer_query('when does flight AF1234 depart from Barcelona?')


Query: "when does flight AF1234 depart from Barcelona?"

[STAGE 1] Clarification agent...
429 rate-limited. Waiting 5s (attempt 1/3)...
  Score: 0.95 [CLEAR]
  Reasoning: The flight number, departure city, and intent are all clearly specified.

[STAGE 2] Retrieval agent...
  [0.853] FL-001: Flight AF1234 from Barcelona to Paris departs daily at 07:15 CET from Terminal 1...
  [0.731] FL-002: Flight IB2050 from Barcelona to Madrid departs daily at 09:30 CET from Terminal ...
  [0.706] FL-003: Flight LH1811 from Barcelona to Frankfurt departs daily at 14:20 CET from Termin...

[STAGE 3] Generation agent...
  Answer: Flight AF1234 departs from Barcelona daily at 07:15 CET [FL-001].

[STAGE 4] LLM-as-Judge...
  Faithfulness:  1.00
  Relevance:     1.00
  Groundedness:  1.00
  Verdict:       PASS
  Reasoning:     The answer directly and accurately addresses the query using information from the provided source with correct citation.


**Scenario 3 — schedule query, and my resilience layer proving itself.** Watch the log line `429 rate-limited. Waiting 5s (attempt 1/3)` — this is my backoff wrapper catching a live throttle from the shared endpoint, waiting the server-suggested delay, and succeeding on retry. The query then resolves cleanly to FL-001 with a PASS. This is the single best demonstration in the notebook that I designed for the realities of hosted inference rather than assuming an infinitely-available API.

In [21]:
# Scenario 4: Ambiguous flight query — should ask for flight number
_ = answer_query('what time does my flight leave?')


Query: "what time does my flight leave?"

[STAGE 1] Clarification agent...
  Score: 0.30 [AMBIGUOUS]
  Reasoning: The query lacks a flight number or booking reference to look up the departure time.
  --> Below threshold (0.7). Asking for clarification.

Final response:
I need a bit more information to help you:
  - What is your flight number or booking reference?
  - What date is your flight?


**Scenario 4 — ambiguous flight, correct clarification.** `"what time does my flight leave?"` scores **0.30**: the intent is clear but the *entity* (flight number / booking reference) is missing. The system asks for exactly that. This contrasts nicely with Scenario 3 — same domain, but here the guardrail correctly refuses to guess which flight the passenger means.

In [22]:
# Scenario 5: Out of domain — should hedge
_ = answer_query('what is the meaning of life?')


Query: "what is the meaning of life?"

[STAGE 1] Clarification agent...
  Score: 0.00 [AMBIGUOUS]
  Reasoning: The query is completely unrelated to airline policies, schedules, or services.
  --> Below threshold (0.7). Asking for clarification.

Final response:
I need a bit more information to help you:
  - How can I assist you with your flight, baggage, or travel plans?


**Scenario 5 — out-of-domain, caught at the door.** `"what is the meaning of life?"` scores **0.00** and is stopped at Stage 1. Worth highlighting: the out-of-domain case is handled by the *clarification* guardrail, before retrieval — so I never even reach the "low retrieval score" fallback. It's a clean example of defence-in-depth: the earliest, cheapest guardrail already covers the case.

## 11. Conclusions & interview notes

### What I built
A **multi-agent RAG pipeline** for flight-passenger queries, running on Google Gemini, composed of four cooperating agents behind one orchestrator:
1. **Clarification agent** — an *input guardrail* that scores answerability and asks back on ambiguous or out-of-domain queries, before spending any retrieval/generation budget.
2. **Retrieval agent** — semantic top-k search over a FAISS vector store with asymmetric document/query embeddings.
3. **Generation agent** — source-grounded, citation-bearing answering (no free recall).
4. **LLM-as-Judge** — an *evaluation layer* scoring faithfulness, relevance and groundedness, flagging weak answers for human review.

### What the demo actually proved
- Ambiguous (S1), out-of-domain (S5) and missing-entity (S4) queries **all exit early at the clarification stage** — the guardrail works and is cheap.
- Clear queries (S2, S3) run the full retrieve → generate → judge chain and return **cited, PASS-rated** answers.
- S3 caught a **live 429 and recovered via backoff** — evidence the system is built for the realities of a shared, rate-limited inference endpoint, not a perfect API.

### Infrastructure themes I want to talk through
- **Separation of concerns** — embedding model, generation model, retrieval store and orchestration are independently swappable. That's what makes the Azure port mechanical, not a rewrite.
- **Designing for a serving layer I don't own** — model deprecation, `quota: 0`, region availability and throttling are *expected* conditions. Treating the model id as config and centralising retry/backoff is how I handle them.
- **Guardrails at the cheapest point** — validate the query at the edge (clarification) rather than letting bad input propagate into retrieval and generation.
- **Evaluation as infrastructure** — the judge and the per-stage `stages` trace are the seams for offline eval, online monitoring and drift detection.

### Honest limitations & what I'd harden next (in priority order)
1. **Compute the judge verdict in my own code**, not from the LLM — I observed it emit a numeric verdict instead of `PASS`/`FAIL`, which would let a bad answer slip through. Threshold logic belongs in code; the LLM only supplies the scores.
2. **Deterministic groundedness check** — verify cited ids ⊆ retrieved ids with plain code before trusting the LLM's groundedness score.
3. **A stronger / different judge model** than the generator, to break correlated blind spots in self-evaluation.
4. **Adversarial judge tests** — feed a known-hallucinated answer and confirm the judge *fails* it; right now every score is 1.0, so the judge is asserted, not proven.
5. **Retrieval score floor** — a secondary out-of-domain guard for queries that slip past clarification.
6. **Secrets & scale** — move the API key to a secret store (never inline), and swap FAISS-Flat for a managed ANN service as the corpus grows.

### Azure mapping (production at AXA)
| Demo | Production |
|------|-----------|
| Gemini `gemini-flash-latest` | Azure OpenAI GPT-4o |
| `gemini-embedding-001` | Azure OpenAI `text-embedding-3` |
| FAISS `IndexFlatIP` | Azure AI Search (hybrid + semantic ranking) |
| Inline API key | Azure Key Vault |
| This notebook | Azure Functions + Azure OpenAI + Azure AI Search |
| `stages` trace | App Insights / Azure Monitor (latency, judge scores, drift) |

**Cost:** with Flash-class models this runs at fractions of a cent per query; the clarification early-exit further cuts cost by skipping retrieval+generation on unanswerable queries.